# MVP UI Demo: Full Cycle Entry Point

Этот notebook — каноническая пользовательская точка входа поверх полного пайплайна `run_full_cycle` только по листу `"Общее"`.

- `MODE = "full"` — полное переобучение и пересборка артефактов.
- `MODE = "fast"` — повторный запуск с уже сохраненной моделью.

Базовый сценарий: `Restart Kernel` -> `Run All`.


In [ ]:
from pathlib import Path

from hidden_patterns_combat.app.input_locator import resolve_input_excel

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Project root not found (expected pyproject.toml and src/).")

PROJECT_ROOT = find_project_root(Path.cwd())
input_resolution = resolve_input_excel(PROJECT_ROOT)
INPUT_PATH = input_resolution.input_path

OUTPUT_DIR = PROJECT_ROOT / "artifacts" / "demo_run"
MODEL_PATH = OUTPUT_DIR / "models" / "hmm_model.pkl"

SHEET_NAMES = ["Общее"]
HEADER_DEPTH = 4
MODE = "full"  # "full" | "fast"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"INPUT_PATH: {INPUT_PATH}")
print(f"INPUT_SOURCE: {input_resolution.source}")
print("Checked input candidates:")
for candidate in input_resolution.checked_paths:
    print(f" - {candidate}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"MODEL_PATH: {MODEL_PATH}")
print(f"SHEET_NAMES: {SHEET_NAMES}")
print(f"HEADER_DEPTH: {HEADER_DEPTH}")
print(f"MODE: {MODE}")


In [ ]:
# Optional: ручная очистка перед полным прогоном.
# По умолчанию удаление отключено, чтобы `Run All` был безопасным.

import shutil

RUN_CLEANUP = False  # Переключите в True и запустите ячейку вручную при необходимости

if RUN_CLEANUP:
    if OUTPUT_DIR.exists():
        shutil.rmtree(OUTPUT_DIR)
        print(f"Removed: {OUTPUT_DIR}")
    else:
        print(f"Nothing to remove: {OUTPUT_DIR}")
else:
    print("Cleanup skipped (RUN_CLEANUP=False).")


In [ ]:
from hidden_patterns_combat.app.full_cycle import run_full_cycle

result = run_full_cycle(
    input_path=INPUT_PATH,
    output_dir=OUTPUT_DIR,
    sheet_names=SHEET_NAMES,
    header_depth=HEADER_DEPTH,
    mode=MODE,
    model_path=MODEL_PATH,
)

result.as_dict()


In [ ]:
import json
import pandas as pd
from IPython.display import Markdown, display

diagnostics_csv = OUTPUT_DIR / "diagnostics" / "hmm_state_interpretation.csv"
semantic_json = OUTPUT_DIR / "diagnostics" / "semantic_diagnostics.json"
report_md = OUTPUT_DIR / "reports" / "full_cycle_report.md"

payload = result.as_dict()
mapping = payload.get("canonical_state_mapping", {})
observed_signal = payload.get("observed_signal", {})

print("Canonical state order:", payload.get("canonical_state_order"))
print("Canonical state mapping:", mapping)
print("Semantic assignment:", payload.get("semantic_assignment"))
print("Semantic confidence:", payload.get("semantic_confidence"))
print("Transition alignment:", payload.get("transition_alignment"))
print("Semantic order matched topology before reorder:", payload.get("semantic_order_matches_topology_before_reorder"))

print("Observed signal block:")
print(json.dumps(observed_signal, ensure_ascii=False, indent=2))
if not observed_signal.get("is_direct_observed_zap", False):
    print("WARNING: observed signal is proxy, not direct observed ZAP.")

warnings = payload.get("semantic_warnings", [])
if warnings:
    print("Warnings:")
    for w in warnings:
        print(" -", w)
else:
    print("Warnings: none")


consistency_warnings = payload.get("consistency_warnings", [])
if consistency_warnings:
    print("Consistency warnings:")
    for w in consistency_warnings:
        print(" -", w)
else:
    print("Consistency warnings: none")
transition_summary = payload.get("transition_summary", [])
if transition_summary:
    print("Top transitions:")
    display(pd.DataFrame(transition_summary).head(10))
else:
    print("Transition summary is empty.")

if diagnostics_csv.exists():
    print(f"Diagnostics: {diagnostics_csv}")
    display(pd.read_csv(diagnostics_csv))
else:
    print(f"Diagnostics file not found: {diagnostics_csv}")

if semantic_json.exists():
    print(f"Semantic diagnostics: {semantic_json}")
    display(Markdown(f"```json\n{semantic_json.read_text(encoding='utf-8')}\n```"))
else:
    print(f"Semantic diagnostics file not found: {semantic_json}")

if report_md.exists():
    print(f"Report: {report_md}")
    display(Markdown(report_md.read_text(encoding="utf-8")))
else:
    print(f"Report file not found: {report_md}")


In [ ]:
from IPython.display import Image, display

plots_dir = OUTPUT_DIR / "plots"
expected_plots = [
    "hidden_state_sequence.png",
    "state_probability_profile.png",
    "transition_distribution.png",
    "athlete_comparative_profile.png",
    "scenario_success_frequencies.png",
]

shown = set()

if plots_dir.exists():
    for name in expected_plots:
        path = plots_dir / name
        if path.exists():
            print(f"Showing: {path}")
            display(Image(filename=str(path)))
            shown.add(path.name)
        else:
            print(f"Missing plot (skip): {path}")

    for extra_path in sorted(plots_dir.glob("*.png")):
        if extra_path.name not in shown:
            print(f"Showing extra plot: {extra_path}")
            display(Image(filename=str(extra_path)))
else:
    print(f"Plots directory not found: {plots_dir}")


In [ ]:
print("Run completed. Artifacts:")
print(f"- Model: {MODEL_PATH}")
print(f"- Diagnostics: {OUTPUT_DIR / 'diagnostics'}")
print(f"- Plots: {OUTPUT_DIR / 'plots'}")
print(f"- Report: {OUTPUT_DIR / 'reports' / 'full_cycle_report.md'}")
